In [1]:
## This notebook will be running simultaneously to the wikipedia data downloading

In [2]:
from pathlib import Path
import os

workspace = Path("/workspace")

data_root = workspace / "multilingual_medical_bert_data"

raw_root = data_root / "raw"
processed_root = data_root / "processed"
metadata_root = data_root / "metadata"

wikimedia_raw = raw_root / "wikimedia"
wikipedia_processed = processed_root / "wikipedia"

wikipedia_processed.mkdir(
    parents=True,
    exist_ok=True
)

print("CPUs available :", os.cpu_count())
print("Raw data       :", wikimedia_raw)
print("Processed data :", wikipedia_processed)

CPUs available : 128
Raw data       : /workspace/multilingual_medical_bert_data/raw/wikimedia
Processed data : /workspace/multilingual_medical_bert_data/processed/wikipedia


In [3]:
## Then discover only completed .xml.bz2 files:
dump_files = sorted(
    wikimedia_raw.rglob("*.xml.bz2")
)

print(f"\nCompleted Wikipedia shards: {len(dump_files)}")

total_bytes = sum(
    path.stat().st_size
    for path in dump_files
)

print(
    f"Downloaded/verified data   : "
    f"{total_bytes / 1024**3:.2f} GiB"
)

print("\nFirst few shards:")

for path in dump_files[:10]:
    print(path)


Completed Wikipedia shards: 91
Downloaded/verified data   : 33.86 GiB

First few shards:
/workspace/multilingual_medical_bert_data/raw/wikimedia/abwiki/2026-09-01/abwiki-2026-09-01-p1p55923.xml.bz2
/workspace/multilingual_medical_bert_data/raw/wikimedia/acewiki/2026-09-01/acewiki-2026-09-01-p1p33735.xml.bz2
/workspace/multilingual_medical_bert_data/raw/wikimedia/adywiki/2026-09-01/adywiki-2026-09-01-p4p5969.xml.bz2
/workspace/multilingual_medical_bert_data/raw/wikimedia/afwiki/2026-09-01/afwiki-2026-09-01-p2p468468.xml.bz2
/workspace/multilingual_medical_bert_data/raw/wikimedia/alswiki/2026-09-01/alswiki-2026-09-01-p1p94943.xml.bz2
/workspace/multilingual_medical_bert_data/raw/wikimedia/altwiki/2026-09-01/altwiki-2026-09-01-p3p7701.xml.bz2
/workspace/multilingual_medical_bert_data/raw/wikimedia/amiwiki/2026-09-01/amiwiki-2026-09-01-p3p3251.xml.bz2
/workspace/multilingual_medical_bert_data/raw/wikimedia/amwiki/2026-09-01/amwiki-2026-09-01-p1p55893.xml.bz2
/workspace/multilingual_medica

In [4]:
## Insepecting XML structure and building a streaming parser
import bz2
import xml.etree.ElementTree as ET

sample_file = dump_files[0]

print("Inspecting:")
print(sample_file)

with bz2.open(sample_file, "rb") as f:

    context = ET.iterparse(
        f,
        events=("start", "end")
    )

    for event, elem in context:

        # Remove XML namespace for readability
        tag = elem.tag.split("}")[-1]

        if event == "end" and tag == "page":

            print("\nFirst <page> found\n")
            print("Children:")

            for child in elem:
                print(
                    child.tag.split("}")[-1],
                    "→",
                    repr(child.text)[:150]
                )

            break

Inspecting:
/workspace/multilingual_medical_bert_data/raw/wikimedia/abwiki/2026-09-01/abwiki-2026-09-01-p1p55923.xml.bz2

First <page> found

Children:
title → 'Main Page'
ns → '0'
id → '1'
redirect → None
revision → '\n      '


In [5]:
## The above confirms the standard MediaWiki page structure
## extract actual article records

def extract_pages(xml_bz2_path, max_pages=None):

    pages_found = 0

    with bz2.open(xml_bz2_path, "rb") as f:

        context = ET.iterparse(
            f,
            events=("end",)
        )

        for event, elem in context:

            tag = elem.tag.split("}")[-1]

            if tag != "page":
                continue

            # -------------------------
            # Basic page metadata
            # -------------------------

            title = None
            namespace = None
            page_id = None
            redirect = False
            text = None

            for child in elem:

                child_tag = child.tag.split("}")[-1]

                if child_tag == "title":
                    title = child.text

                elif child_tag == "ns":
                    namespace = child.text

                elif child_tag == "id" and page_id is None:
                    page_id = child.text

                elif child_tag == "redirect":
                    redirect = True

                elif child_tag == "revision":

                    for revision_child in child:

                        revision_tag = (
                            revision_child.tag
                            .split("}")[-1]
                        )

                        if revision_tag == "text":
                            text = revision_child.text

            # -------------------------
            # Keep actual articles
            # -------------------------

            if (
                namespace == "0"
                and not redirect
                and text
            ):

                yield {
                    "page_id": page_id,
                    "title": title,
                    "text": text
                }

                pages_found += 1

                if (
                    max_pages is not None
                    and pages_found >= max_pages
                ):
                    return

            # Critical for streaming:
            # release processed XML from memory
            elem.clear()

In [6]:
for page in extract_pages(
    sample_file,
    max_pages=3
):

    print("=" * 100)

    print("PAGE ID:", page["page_id"])
    print("TITLE  :", page["title"])

    print("\nRAW WIKITEXT:\n")

    print(page["text"][:1500])

    print()

PAGE ID: 807
TITLE  : Аԥсуа бызшәа

RAW WIKITEXT:

{{Акарточка абызшәа
| ахьӡ             = Аԥсуа бызшәа
| даҽа ахьӡ        = Аԥсшәа
| ахатәы хьӡы      = {{lang|ab|Аԥсуа бызшәа}}
| афото            = 
| imagescale       = 1
| imagecaption     = 
| ацәажәашьа       = 
| атәылақәа        = {{hlist|style=line-height:1.3em;
 | [[Аԥсны Аҳәынҭқарра|Аԥсны]]
 | ''[[Аԥснытәи Автономтә Ареспублика|Аԥснытәи Автономтә Ареспублика]]'' }}
| ацәажәаҩцәа      = 190 000
| арыцхә           = 2015-2019
| ref              = 
| ацәажәаҩцәа2     = 
| familycolor      = Caucasian
| аҭа1             = [[Аҩада-мраҭашәаратәи Кавказтәи ҭаацәара бызшәақәа|Аҩада-мраҭашәаратәи Кавказтәи ҭаацәара]]
| аҭа2             = [[Абаза бызшәа|Абаза]]–Аԥсуа
| listclass        = hlist
| адиа1            = [[Абжьыуа диалект]] 
| адиа2            = [[Бзып диалект]] 
| адиа3            = [[Асаӡ диалект]] 
| адиа4            = [[Атапанҭатәи диалект]] 
| аҩыратә система  = [[Акириллица]]
| аҳәынҭқарратә бызшәа = [[Аԥсны Аҳәынҭқарра

In [7]:
#%pip install -q mwparserfromhell

In [8]:
import mwparserfromhell
import re
import unicodedata

In [9]:
def clean_wikitext(text):

    if not text:
        return ""

    # ---------------------------------------
    # Parse MediaWiki markup
    # ---------------------------------------

    code = mwparserfromhell.parse(text)

    # Remove templates:
    # {{Infobox ...}}
    # {{citation ...}}
    # {{lang ...}}
    # etc.
    for template in code.filter_templates(
        recursive=True
    ):
        try:
            code.remove(template)
        except ValueError:
            pass

    # Remove references/tags
    for tag in code.filter_tags(
        recursive=True
    ):
        try:
            code.remove(tag)
        except ValueError:
            pass

    # Convert remaining markup to text
    text = code.strip_code(
        normalize=True,
        collapse=True
    )

    # ---------------------------------------
    # Unicode normalization
    # ---------------------------------------

    text = unicodedata.normalize(
        "NFC",
        text
    )

    # ---------------------------------------
    # Whitespace cleanup
    # ---------------------------------------

    text = text.replace("\xa0", " ")

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\n[ \t]+",
        "\n",
        text
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()

In [10]:
for page in extract_pages(
    sample_file,
    max_pages=3
):

    cleaned = clean_wikitext(
        page["text"]
    )

    print("=" * 100)

    print("PAGE ID :", page["page_id"])
    print("TITLE   :", page["title"])

    print("\nCLEAN TEXT:\n")

    print(cleaned[:2000])

    print()

PAGE ID : 807
TITLE   : Аԥсуа бызшәа

CLEAN TEXT:

, (, апсашва, , ) — Аԥсны ҳәынҭқарра бызшәас иамоуп. Ихалхно — 125 000 ҩык.

Аԥсуа бызшәа амш — жьҭаарамза 27.

Аԥсуа алфавит

Аграматика

Абжьыҟатә шьҭыбжьаҟәа
[ā] ← [ʕ]
[a] 
[ə] 
[ø̞] ← [ə] ()
[ʏ] ← [ɥˤʏ] ← [ɥˤə] 
[e̞] ← [je̞] ← [ja] 
[e̞] ← [e̞j] ← [aj] 
[i] ← [əj] 
[i] ← [jə] 
[o̞] ← [wo̞] ← уа [wa] 
[o̞] ← [o̞w] ← ау [aw] 
[u] ← [əw] [wə] 

Аҵыбжьыҟатә шьҭыбжьаҟәа

Ацыбжьыҟақәа уԥылоит бзыԥтәи адиалект мцараҿ, ԥшшәыла иазгәаҭо бзыԥтәи, асаӡтәи диалектқәа — , абжьауатәи — . 
Авелиартә фрикатақәа иреиԥшхаӡом ахархәара аҭыԥ зеиԥшро ала, авелиартәи ма аувелиартәи, иахьатәи аамҭаз бзыԥтәы адиалект аҿы ахархәара амоуп афрингалтә шьҭабжьқәа /χˁ χˁʷ/.
Ахәлымшәытә еидылара [ʔ] ицәырнагоит, актәи ахархәаратә феномен аҟынтә /q’/, аҩбатәи, иара убас иуԥылоит ажәаҿ /ʔaj/ «мап». /ɥ/ уи иахылҵуеит аҟырҟы еиҵалара /ʕʷ/, иара убас уи уаҳауеит анхарҭа ҭыԥқәакырҿ. 
Афонологиатә қәиӷәӷәара аҵакы амоуп.

Аҩышьаԥҟарақәа

Аԥсуа жәақәа ажәытәан рынҵрақәа

In [11]:
import mwparserfromhell
import re
import unicodedata


TEXT_TEMPLATES = {
    "lang",
    "lang-ab",
    "lang-abq",
    "lang-ar",
    "lang-de",
    "lang-en",
    "lang-es",
    "lang-fa",
    "lang-fr",
    "lang-hi",
    "lang-ja",
    "lang-ka",
    "lang-ko",
    "lang-ru",
    "lang-zh",
    "transl",
    "transliteration",
}


def clean_wikitext(text):

    if not text:
        return ""

    code = mwparserfromhell.parse(text)

    # -----------------------------------
    # Handle templates
    # -----------------------------------

    for template in code.filter_templates(
        recursive=True
    ):

        name = str(template.name).strip().lower()

        try:

            # {{lang|ab|text}}
            if name == "lang" and len(template.params) >= 2:

                replacement = str(
                    template.params[-1].value
                ).strip()

                code.replace(
                    template,
                    replacement
                )

            # {{lang-ka|text}}, {{lang-ru|text}}, etc.
            elif name.startswith("lang-") and template.params:

                replacement = str(
                    template.params[-1].value
                ).strip()

                code.replace(
                    template,
                    replacement
                )

            # Other templates -> remove
            else:
                code.remove(template)

        except ValueError:
            pass

    # -----------------------------------
    # Remove HTML/ref tags
    # -----------------------------------

    for tag in code.filter_tags(recursive=True):

        try:
            code.remove(tag)
        except ValueError:
            pass

    # -----------------------------------
    # Strip remaining wiki markup
    # -----------------------------------

    text = code.strip_code(
        normalize=True,
        collapse=True
    )

    # -----------------------------------
    # Remove image/file remnants
    # -----------------------------------

    text = re.sub(
        r"\b\d+px\s*\|?[^\n]*",
        " ",
        text
    )

    text = re.sub(
        r"\b(?:мини|thumb|thumbnail|center|centre|центр)\s*\|",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # -----------------------------------
    # Unicode normalization
    # -----------------------------------

    text = unicodedata.normalize(
        "NFC",
        text
    )

    text = text.replace("\xa0", " ")

    # -----------------------------------
    # Whitespace
    # -----------------------------------

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r" *\n *",
        "\n",
        text
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()

In [12]:
for page in extract_pages(
    sample_file,
    max_pages=3
):

    cleaned = clean_wikitext(
        page["text"]
    )

    print("=" * 100)
    print("PAGE ID :", page["page_id"])
    print("TITLE   :", page["title"])
    print("\nCLEAN TEXT:\n")
    print(cleaned[:2000])
    print()

PAGE ID : 807
TITLE   : Аԥсуа бызшәа

CLEAN TEXT:

, (апс(ы)уа бызшва, апсашва, абхазский язык, аbhaz dili, аbaza dili) — Аԥсны ҳәынҭқарра бызшәас иамоуп. Ихалхно — 125 000 ҩык.

Аԥсуа бызшәа амш — жьҭаарамза 27.

Аԥсуа алфавит

Аграматика

Абжьыҟатә шьҭыбжьаҟәа
[ā] ← [ʕ]
[a]
[ə]
[ø̞] ← [ə] ()
[ʏ] ← [ɥˤʏ] ← [ɥˤə]
[e̞] ← [je̞] ← [ja]
[e̞] ← [e̞j] ← [aj]
[i] ← [əj]
[i] ← [jə]
[o̞] ← [wo̞] ← уа [wa]
[o̞] ← [o̞w] ← ау [aw]
[u] ← [əw] [wə]

Аҵыбжьыҟатә шьҭыбжьаҟәа

Ацыбжьыҟақәа уԥылоит бзыԥтәи адиалект мцараҿ, ԥшшәыла иазгәаҭо бзыԥтәи, асаӡтәи диалектқәа — , абжьауатәи — .
Авелиартә фрикатақәа иреиԥшхаӡом ахархәара аҭыԥ зеиԥшро ала, авелиартәи ма аувелиартәи, иахьатәи аамҭаз бзыԥтәы адиалект аҿы ахархәара амоуп афрингалтә шьҭабжьқәа /χˁ χˁʷ/.
Ахәлымшәытә еидылара [ʔ] ицәырнагоит, актәи ахархәаратә феномен аҟынтә /q’/, аҩбатәи, иара убас иуԥылоит ажәаҿ /ʔaj/ «мап». /ɥ/ уи иахылҵуеит аҟырҟы еиҵалара /ʕʷ/, иара убас уи уаҳауеит анхарҭа ҭыԥқәакырҿ.
Афонологиатә қәиӷәӷәара аҵакы амоуп.

Аҩышьаԥҟ

In [13]:
import bz2
import gzip
import json
import os
import shutil
import time
from pathlib import Path


def process_wikipedia_shard(
    input_path,
    checkpoint_every=1000
):

    start_time = time.time()

    wiki_id = input_path.parent.parent.name
    snapshot = input_path.parent.name

    output_dir = (
        wikipedia_processed
        / wiki_id
        / snapshot
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    base_name = input_path.name.replace(
        ".xml.bz2",
        ""
    )

    final_path = (
        output_dir /
        f"{base_name}.jsonl.gz"
    )

    part_path = (
        output_dir /
        f"{base_name}.jsonl.part"
    )

    checkpoint_path = (
        output_dir /
        f"{base_name}.checkpoint.json"
    )

    # -----------------------------------
    # Already completely processed
    # -----------------------------------

    if final_path.exists():

        return {
            "wiki_id": wiki_id,
            "input": str(input_path),
            "output": str(final_path),
            "status": "already_processed"
        }

    # -----------------------------------
    # Load checkpoint
    # -----------------------------------

    resume_after_page_id = None
    articles_written = 0
    characters_written = 0

    if (
        part_path.exists()
        and checkpoint_path.exists()
    ):

        with open(
            checkpoint_path,
            "r",
            encoding="utf-8"
        ) as f:

            checkpoint = json.load(f)

        resume_after_page_id = (
            checkpoint["last_page_id"]
        )

        articles_written = (
            checkpoint["articles_written"]
        )

        characters_written = (
            checkpoint["characters_written"]
        )

        file_mode = "a"

    else:

        # Inconsistent leftovers should not be resumed
        if part_path.exists():
            part_path.unlink()

        if checkpoint_path.exists():
            checkpoint_path.unlink()

        file_mode = "w"

    # -----------------------------------
    # Stream XML
    # -----------------------------------

    resume_reached = (
        resume_after_page_id is None
    )

    pages_seen = 0
    pages_since_checkpoint = 0

    with open(
        part_path,
        file_mode,
        encoding="utf-8"
    ) as fout:

        for page in extract_pages(input_path):

            pages_seen += 1

            # -----------------------------------
            # Resume
            # -----------------------------------

            if not resume_reached:

                if (
                    page["page_id"]
                    == resume_after_page_id
                ):
                    resume_reached = True

                continue

            # -----------------------------------
            # Clean
            # -----------------------------------

            clean_text = clean_wikitext(
                page["text"]
            )

            if len(clean_text) >= 200:

                record = {
                    "id": f"{wiki_id}_{page['page_id']}",
                    "source": "wikipedia",
                    "wiki_id": wiki_id,
                    "snapshot": snapshot,
                    "page_id": page["page_id"],
                    "title": page["title"],
                    "text": clean_text
                }

                fout.write(
                    json.dumps(
                        record,
                        ensure_ascii=False
                    )
                    + "\n"
                )

                articles_written += 1
                characters_written += len(
                    clean_text
                )

            pages_since_checkpoint += 1

            # -----------------------------------
            # Checkpoint
            # -----------------------------------

            if (
                pages_since_checkpoint
                >= checkpoint_every
            ):

                fout.flush()
                os.fsync(fout.fileno())

                checkpoint = {
                    "last_page_id": page["page_id"],
                    "articles_written": articles_written,
                    "characters_written": characters_written
                }

                temp_checkpoint = Path(
                    str(checkpoint_path) + ".tmp"
                )

                with open(
                    temp_checkpoint,
                    "w",
                    encoding="utf-8"
                ) as f:

                    json.dump(
                        checkpoint,
                        f
                    )

                temp_checkpoint.replace(
                    checkpoint_path
                )

                pages_since_checkpoint = 0

    # -----------------------------------
    # Compress completed shard
    # -----------------------------------

    temp_gzip = Path(
        str(final_path) + ".tmp"
    )

    with open(part_path, "rb") as fin:

        with gzip.open(
            temp_gzip,
            "wb",
            compresslevel=6
        ) as fout:

            shutil.copyfileobj(
                fin,
                fout,
                length=8 * 1024 * 1024
            )

    # Atomic completion
    temp_gzip.replace(final_path)

    # -----------------------------------
    # Remove recovery files
    # -----------------------------------

    part_path.unlink(
        missing_ok=True
    )

    checkpoint_path.unlink(
        missing_ok=True
    )

    elapsed = time.time() - start_time

    return {
        "wiki_id": wiki_id,
        "input": str(input_path),
        "output": str(final_path),
        "status": "processed",
        "articles_written": articles_written,
        "characters": characters_written,
        "seconds": elapsed
    }

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
import os

# Refresh list — downloader may have completed more shards
dump_files = sorted(
    wikimedia_raw.rglob("*.xml.bz2")
)

print("CPU threads available :", os.cpu_count())
print("Wikipedia shards ready:", len(dump_files))

max_workers = 12

processing_results = []

with ProcessPoolExecutor(
    max_workers=max_workers
) as executor:

    futures = {
        executor.submit(
            process_wikipedia_shard,
            path
        ): path
        for path in dump_files
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Processing Wikipedia"
    ):

        path = futures[future]

        try:
            result = future.result()

        except Exception as e:
            result = {
                "input": str(path),
                "status": "failed",
                "error": repr(e)
            }

        processing_results.append(result)


processing_df = pd.DataFrame(
    processing_results
)

processing_status_path = (
    metadata_root /
    "wikipedia_processing_status.csv"
)

processing_df.to_csv(
    processing_status_path,
    index=False
)

print("\nProcessing summary:")

print(
    processing_df["status"]
    .value_counts(dropna=False)
)

CPU threads available : 128
Wikipedia shards ready: 92


Processing Wikipedia:   0%|          | 0/92 [00:00<?, ?it/s]

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm
import pandas as pd
import os

# Refresh because more downloads may have completed
dump_files = sorted(
    wikimedia_raw.rglob("*.xml.bz2")
)

print("CPU threads available :", os.cpu_count())
print("Shards currently ready:", len(dump_files))

max_workers = 12

processing_results = []

with ProcessPoolExecutor(
    max_workers=max_workers
) as executor:

    futures = {
        executor.submit(
            process_wikipedia_shard,
            path
        ): path

        for path in dump_files
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Processing Wikipedia shards"
    ):

        path = futures[future]

        try:
            result = future.result()

        except Exception as e:
            result = {
                "input": str(path),
                "status": "failed",
                "error": str(e)
            }

        processing_results.append(result)


processing_df = pd.DataFrame(
    processing_results
)

processing_status_path = (
    metadata_root /
    "wikipedia_processing_status.csv"
)

processing_df.to_csv(
    processing_status_path,
    index=False
)

print("\nProcessing summary:")
print(
    processing_df["status"]
    .value_counts(dropna=False)
)

CPU threads available : 128
Shards currently ready: 83


Processing Wikipedia shards:   0%|          | 0/83 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path

part_files = list(
    wikipedia_processed.rglob("*.part")
)

print("Active output files:", len(part_files))

for path in part_files[:20]:
    print(
        f"{path.name:70s}",
        f"{path.stat().st_size / 1024**2:8.2f} MiB"
    )